# Feature Selection: Picking the Right Features

**Notebook Goal:** Understand why and how to select the most relevant features for machine learning models, reducing noise, improving generalization, and speeding up training.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression, load_diabetes, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score, cross_validate,
                                     RepeatedStratifiedKFold, KFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, chi2, f_classif, mutual_info_classif,
    RFE, RFECV, SelectFromModel, SequentialFeatureSelector
)
from sklearn.linear_model import LogisticRegression, Lasso, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from sklearn.svm import SVC

sns.set_style('whitegrid')
np.random.seed(42)
print('All imports successful.')

## 1. Why Feature Selection?

**The Curse of Dimensionality:** As the number of features grows, the data becomes sparse in high-dimensional space, making it exponentially harder to find meaningful patterns. Models require exponentially more samples to generalize.

**Interpretability:** Simpler models with fewer features are easier to explain, debug, and trust — critical in regulated industries (finance, healthcare).

**Overfitting:** Irrelevant or redundant features provide noise that the model may latch onto during training, hurting generalization on unseen data.

**Computational Cost:** Fewer features mean faster training, lower memory usage, and cheaper deployment.

In [ ]:
# Demonstrate curse of dimensionality conceptually
dims = np.arange(1, 101)
volume_fraction = (1 / (2 * np.sqrt(3))) ** dims  # fraction of points in a unit hypercube

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(dims, np.log(volume_fraction), color='crimson', lw=2)
ax[0].set_title('Curse of Dimensionality', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Number of Dimensions')
ax[0].set_ylabel('Log(Fraction of Volume in Hypercube)')
ax[0].axhline(np.log(0.01), ls='--', color='gray', alpha=0.6)
ax[0].text(5, np.log(0.01) + 0.5, '1% volume', fontsize=9, color='gray')

ax[1].bar(['All Features', 'Selected Features'], [0.92, 0.96], color=['#7f8c8d', '#2ecc71'], width=0.5)
ax[1].set_ylabel('Test Accuracy')
ax[1].set_title('Impact of Feature Selection on Performance', fontsize=14, fontweight='bold')
ax[1].set_ylim(0.85, 1.0)
for i, v in enumerate([0.92, 0.96]):
    ax[1].text(i, v + 0.005, f'{v:.0%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Filter Methods

Filter methods evaluate features independently of any model using statistical measures. They are fast, scalable, and provide a solid first pass.

### 2.1 Variance Threshold

Removes features with variance below a threshold — features that are nearly constant carry little information.

In [ ]:
# Variance Threshold demo
rng = np.random.default_rng(42)
X_var = pd.DataFrame({
    'feat_a': rng.normal(0, 1, 200),
    'feat_b': rng.normal(0, 0.01, 200),   # very low variance — candidate to drop
    'feat_c': rng.normal(0, 5, 200),
    'feat_d': np.zeros(200),                # zero variance
    'feat_e': rng.uniform(0, 1, 200)
})

print('Feature variances:')
print(X_var.var().to_string())

selector = VarianceThreshold(threshold=0.05)
X_selected = selector.fit_transform(X_var)
mask = selector.get_support()
print(f'\nFeatures retained after VarianceThreshold(0.05): {list(X_var.columns[mask])}')
print(f'Features dropped: {list(X_var.columns[~mask])}')

### 2.2 Pearson Correlation (with Target)

Measures linear relationship between each feature and the target. Features with very low absolute correlation are weak predictors.
Also useful for removing **multicollinearity** — highly correlated features (>0.9) can destabilize linear models.

In [ ]:
# Correlation analysis on breast cancer dataset
data = load_breast_cancer()
X_cancer = pd.DataFrame(data.data, columns=data.feature_names)
y_cancer = data.target

corr_with_target = pd.Series(
    [np.corrcoef(X_cancer[c], y_cancer)[0, 1] for c in X_cancer.columns],
    index=X_cancer.columns
).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71' if v > 0.5 else '#e74c3c' for v in corr_with_target.values]
corr_with_target.head(15).plot(kind='barh', color=colors[::-1], ax=ax, edgecolor='white')
ax.set_title('Top 15 Features — Absolute Pearson Correlation with Target', fontsize=13, fontweight='bold')
ax.set_xlabel('|Correlation|')
ax.axvline(0.5, ls='--', color='gray', alpha=0.5, label='Threshold = 0.5')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Number of features with |corr| > 0.5: {(corr_with_target > 0.5).sum()} / {len(corr_with_target)}')

In [ ]:
# Correlation heatmap
corr_matrix = X_cancer.corr()

# Find highly correlated pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[col, row])
             for col in upper.columns for row in upper.index
             if not pd.isna(upper.loc[col, row]) and abs(upper.loc[col, row]) > 0.9]

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.3, cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Feature Correlation Heatmap (Breast Cancer)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Highly correlated feature pairs (|r| > 0.9): {len(high_corr)}')
for col, row, val in high_corr[:5]:
    print(f'  {col}  vs  {row}  →  r = {val:.3f}')

### 2.3 Chi-Square Test

Tests independence between categorical features and a categorical target. A high chi-square statistic indicates the feature is not independent of the target.

In [ ]:
# Chi-Square test on categorical-like features
rng = np.random.default_rng(42)
n = 1000
X_cat = pd.DataFrame({
    'color': rng.choice(['red', 'blue', 'green'], n),
    'size': rng.choice(['S', 'M', 'L', 'XL'], n),
    'shape': rng.choice(['round', 'square', 'triangle'], n),
    'noise': rng.choice(['A', 'B'], n)   # irrelevant
})

y_cat = pd.Series(
    ((X_cat['color'] == 'red') | (X_cat['size'] == 'XL')).astype(int)
)

from sklearn.feature_selection import chi2
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')
X_ohe = pd.DataFrame(
    ohe.fit_transform(X_cat),
    columns=ohe.get_feature_names_out(X_cat.columns)
)

chi2_stat, p_val = chi2(X_ohe, y_cat)
chi_results = pd.DataFrame({'chi2_stat': chi2_stat, 'p_value': p_val},
                           index=X_ohe.columns).sort_values('chi2_stat', ascending=False)
print('Chi-Square results (top features):')
print(chi_results.head(8).to_string(float_format='{:.4f}'.format))

### 2.4 ANOVA F-Test

Compares means across groups for numerical features against a categorical target. High F-statistic → feature separates classes well.

In [ ]:
# ANOVA F-test on breast cancer data
f_stat, p_vals = f_classif(X_cancer, y_cancer)
anova_df = pd.DataFrame({'F_statistic': f_stat, 'p_value': p_vals},
                        index=X_cancer.columns).sort_values('F_statistic', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(10), anova_df['F_statistic'].values[:10][::-1], color='#3498db', edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels(anova_df.index[:10][::-1])
ax.set_title('Top 10 Features by ANOVA F-statistic', fontsize=13, fontweight='bold')
ax.set_xlabel('F-statistic')
plt.tight_layout()
plt.show()

### 2.5 Mutual Information

Captures **any** dependency (linear or non-linear) between feature and target. It measures the reduction in uncertainty about the target given the feature.

In [ ]:
# Mutual Information — captures non-linear relationships
mi = mutual_info_classif(X_cancer, y_cancer, random_state=42)
mi_df = pd.DataFrame({'Mutual_Information': mi}, index=X_cancer.columns)\
    .sort_values('Mutual_Information', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(10), mi_df['Mutual_Information'].values[:10][::-1],
             color='#9b59b6', edgecolor='white')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(mi_df.index[:10][::-1])
axes[0].set_title('Mutual Information — Top 10 Features', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mutual Information (nats)')

# Compare MI vs F-test
common = anova_df.join(mi_df, how='inner')
axes[1].scatter(common['F_statistic'], common['Mutual_Information'], alpha=0.7, c='#2ecc71', edgecolors='k')
axes[1].set_xlabel('F-statistic')
axes[1].set_ylabel('Mutual Information')
axes[1].set_title('F-test vs Mutual Information', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Wrapper Methods

Wrapper methods use a **predictive model** to score feature subsets. They are computationally expensive but often yield better subsets than filters.

### 3.1 Forward Selection

Start with an empty set and iteratively add the feature that most improves the model. Stops when no improvement is observed or a desired count is reached.

In [ ]:
# Sequential Feature Selector (Forward) — sklearn's implementation
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=42
)

sfs_forward = SequentialFeatureSelector(
    LogisticRegression(max_iter=2000, random_state=42),
    n_features_to_select='auto',
    tol=1e-4,
    direction='forward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)

selected_forward = list(X_cancer.columns[sfs_forward.get_support()])
print(f'Forward selection selected {len(selected_forward)} features:')
print(selected_forward)
print(f'Accuracy with selected features: '
      f'{accuracy_score(y_test, sfs_forward.estimator_.predict(X_test)):.4f}')

### 3.2 Backward Elimination

Start with all features and iteratively remove the least important one. More stable than forward selection but computationally heavier for high dimensions.

In [ ]:
# Sequential Feature Selector (Backward)
sfs_backward = SequentialFeatureSelector(
    LogisticRegression(max_iter=2000, random_state=42),
    n_features_to_select='auto',
    tol=1e-4,
    direction='backward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)

selected_backward = list(X_cancer.columns[sfs_backward.get_support()])
print(f'Backward elimination selected {len(selected_backward)} features:')
print(selected_backward)

# Compare forward vs backward
common = set(selected_forward) & set(selected_backward)
print(f'\nFeatures selected by both: {len(common)}')
print(f'Forward-only: {set(selected_forward) - common}')
print(f'Backward-only: {set(selected_backward) - common}')

### 3.3 Recursive Feature Elimination (RFE)

Fits a model, ranks features by importance, prunes the least important ones, and repeats until the desired number remains. 
The plot below shows how RFE progressively eliminates features.

In [ ]:
# RFE — visualize the elimination process
estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rfe = RFE(estimator, n_features_to_select=10, step=1)
rfe.fit(X_train, y_train)

ranking = pd.DataFrame({
    'feature': X_cancer.columns,
    'ranking': rfe.ranking_,
    'selected': rfe.support_
}).sort_values('ranking')

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if s else '#e74c3c' for s in ranking['selected']]
ax.barh(range(len(ranking)), ranking['ranking'], color=colors, edgecolor='white')
ax.set_yticks(range(len(ranking)))
ax.set_yticklabels(ranking['feature'])
ax.set_xlabel('RFE Rank (1 = selected, higher = eliminated earlier)')
ax.set_title('RFE Selection Process — Random Forest on Breast Cancer', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Selected'),
                   Patch(facecolor='#e74c3c', label='Eliminated')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print('RFE Accuracy:', accuracy_score(y_test, rfe.predict(X_test)))
print('Selected features:')
print(list(ranking[ranking['selected']]['feature']))

### 3.4 RFECV — RFE with Cross-Validation

Automatically selects the optimal number of features by evaluating performance at each step using cross-validation. 
No need to pre-specify `n_features_to_select`.

In [ ]:
# RFECV — automatic feature count selection
rfecv = RFECV(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    step=1,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    min_features_to_select=5
)
rfecv.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
        rfecv.cv_results_['mean_test_score'], marker='o', color='#3498db', lw=2, markersize=6)
ax.fill_between(
    range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    rfecv.cv_results_['mean_test_score'] - rfecv.cv_results_['std_test_score'],
    rfecv.cv_results_['mean_test_score'] + rfecv.cv_results_['std_test_score'],
    alpha=0.2, color='#3498db'
)
ax.axvline(rfecv.n_features_, color='#e74c3c', ls='--', lw=2,
           label=f'Optimal: {rfecv.n_features_} features')
ax.set_xlabel('Number of Features')
ax.set_ylabel('Cross-Validation Accuracy')
ax.set_title('RFECV — Model Performance vs Number of Features', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Optimal number of features (RFECV): {rfecv.n_features_}')
print(f'CV accuracy at optimum: {rfecv.cv_results_["mean_test_score"][rfecv.n_features_ - 1]:.4f}')
print(f'Test accuracy: {accuracy_score(y_test, rfecv.predict(X_test)):.4f}')

## 4. Embedded Methods

Embedded methods perform feature selection **during** model training. They are generally more efficient than wrapper methods and more expressive than filters.

### 4.1 Lasso (L1 Regularization)

Lasso adds a penalty proportional to the absolute magnitude of coefficients, forcing some to become exactly zero — effectively performing feature selection.

In [ ]:
# Lasso feature selection on regression data
X_reg, y_reg = make_regression(n_samples=200, n_features=30, n_informative=6,
                               noise=15, random_state=42)
feature_names_reg = [f'feat_{i}' for i in range(X_reg.shape[1])]

scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

lasso = Lasso(alpha=0.1, random_state=42, max_iter=10000)
lasso.fit(X_reg_scaled, y_reg)

coef_df = pd.DataFrame({'feature': feature_names_reg, 'coefficient': lasso.coef_})
nonzero = coef_df[coef_df['coefficient'] != 0]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if c != 0 else '#e74c3c' for c in lasso.coef_]
ax.bar(feature_names_reg, lasso.coef_, color=colors, edgecolor='white')
ax.set_title('Lasso Coefficients — Zeroed-out Features in Red', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Features retained by Lasso: {len(nonzero)} / {X_reg.shape[1]}')
print('Non-zero features:', list(nonzero['feature']))

In [ ]:
# Lasso path — coefficients vs alpha
alphas = np.logspace(-3, 1, 50)
coef_path = []
for a in alphas:
    lasso = Lasso(alpha=a, random_state=42, max_iter=10000)
    lasso.fit(X_reg_scaled, y_reg)
    coef_path.append(lasso.coef_)

coef_path = np.array(coef_path)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(coef_path.shape[1]):
    ax.plot(alphas, coef_path[:, i], lw=1, alpha=0.7)
ax.set_xscale('log')
ax.set_xlabel('Alpha (log scale)')
ax.set_ylabel('Coefficient Value')
ax.set_title('Lasso Regularization Path', fontsize=13, fontweight='bold')
ax.axhline(0, color='black', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

### 4.2 Tree-Based Feature Importance

Random Forest and Gradient Boosting provide built-in feature importance scores based on how often a feature is used for splitting and how much it reduces impurity.

In [ ]:
# Tree-based feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

imp_df = pd.DataFrame({
    'feature': X_cancer.columns,
    'Random Forest': rf.feature_importances_,
    'Gradient Boosting': gb.feature_importances_
}).melt(id_vars='feature', var_name='Model', value_name='Importance')

# Top 10 by RF
top10_rf = pd.Series(rf.feature_importances_, index=X_cancer.columns)\
    .sort_values(ascending=False).head(10).index

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=imp_df[imp_df['feature'].isin(top10_rf)],
            x='Importance', y='feature', hue='Model', palette=['#2ecc71', '#3498db'], ax=ax)
ax.set_title('Feature Importance — Random Forest vs Gradient Boosting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 SelectFromModel

A convenient meta-transformer that selects features based on importance weights from any fitted estimator. Works with any model that exposes `coef_` or `feature_importances_`.

In [ ]:
# SelectFromModel with Random Forest
from sklearn.feature_selection import SelectFromModel

sfm = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    threshold='median'
)
sfm.fit(X_train, y_train)

selected_sfm = list(X_cancer.columns[sfm.get_support()])
print(f'SelectFromModel (threshold=median) selected {len(selected_sfm)} features:')
print(selected_sfm)

# Compare with RFE
rfe_selected = list(X_cancer.columns[rfe.support_])
print(f'\nOverlap with RFE: {len(set(selected_sfm) & set(rfe_selected))}')
print(f'Unique to SelectFromModel: {set(selected_sfm) - set(rfe_selected)}')
print(f'Unique to RFE: {set(rfe_selected) - set(selected_sfm)}')

## 5. Feature Selection with Pipelines

Integrating feature selection into a `Pipeline` ensures that selection happens inside the cross-validation loop — preventing **data leakage** and producing honest performance estimates.

In [ ]:
# Pipeline with feature selection
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        threshold='median'
    )),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])

cv_scores = cross_val_score(pipeline, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Compare against no selection
pipeline_no_select = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])
cv_scores_full = cross_val_score(pipeline_no_select, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'Full feature CV accuracy: {cv_scores_full.mean():.4f} ± {cv_scores_full.std():.4f}')

# Extract selected features from pipeline
pipeline.fit(X_cancer, y_cancer)
selected_in_pipe = list(X_cancer.columns[
    pipeline.named_steps['select'].get_support()
])
print(f'\nFeatures selected inside pipeline ({len(selected_in_pipe)}): ')
print(selected_in_pipe)

## 6. Comparing All Methods on a Synthetic Dataset

We create a dataset with 30 features, only 5 of which are informative, to see how each method performs at separating signal from noise.

In [ ]:
# Synthetic dataset with many irrelevant features
X_syn, y_syn = make_classification(
    n_samples=1000, n_features=30, n_informative=5, n_redundant=5,
    n_repeated=0, n_clusters_per_class=2, random_state=42
)
syn_feature_names = [f'feat_{i}' for i in range(X_syn.shape[1])]

X_syn_train, X_syn_test, y_syn_train, y_syn_test = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=42
)

print(f'Shape: {X_syn.shape}')
print(f'Informative features: 5  |  Redundant features: 5  |  Noise features: 20')

In [ ]:
# Define all methods for comparison
def evaluate_method(name, X_tr, X_te, y_tr, y_te, selector, n_to_select=10):
    if name == 'All Features':
        lr = LogisticRegression(max_iter=2000, random_state=42).fit(X_tr, y_tr)
        return accuracy_score(y_te, lr.predict(X_te))

    if hasattr(selector, 'fit_transform'):
        X_tr_sel = selector.fit_transform(X_tr, y_tr)
        X_te_sel = selector.transform(X_te)
    else:
        selector.fit(X_tr, y_tr)
        X_tr_sel = selector.transform(X_tr)
        X_te_sel = selector.transform(X_te)

    lr = LogisticRegression(max_iter=2000, random_state=42)
    lr.fit(X_tr_sel, y_tr_sel)
    return accuracy_score(y_te, lr.predict(X_te_sel))


methods = {
    'All Features': None,
    'SelectKBest (F-test)': SelectKBest(f_classif, k=10),
    'SelectKBest (MI)': SelectKBest(mutual_info_classif, k=10),
    'RFE (RF)': RFE(RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
                    n_features_to_select=10),
    'SelectFromModel (RF)': SelectFromModel(
        RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
        threshold='median'
    ),
    'SelectFromModel (Lasso)': SelectFromModel(
        LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=5000, random_state=42),
        max_features=10
    )
}

results = {}
for name, sel in methods.items():
    if name == 'All Features':
        lr = LogisticRegression(max_iter=2000, random_state=42).fit(X_syn_train, y_syn_train)
        results[name] = accuracy_score(y_syn_test, lr.predict(X_syn_test))
    else:
        X_tr_sel = sel.fit_transform(X_syn_train, y_syn_train)
        X_te_sel = sel.transform(X_syn_test)
        lr = LogisticRegression(max_iter=2000, random_state=42).fit(X_tr_sel, y_syn_train)
        results[name] = accuracy_score(y_syn_test, lr.predict(X_te_sel))

results_df = pd.DataFrame(list(results.items()), columns=['Method', 'Test Accuracy'])
print('Comparison of feature selection methods:')
print(results_df.to_string(index=False))

In [ ]:
# Visual comparison of methods
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(results_df['Method'], results_df['Test Accuracy'],
              color=['#95a5a6', '#3498db', '#9b59b6', '#e74c3c', '#2ecc71', '#f39c12'],
              edgecolor='white', width=0.6)
ax.set_ylabel('Test Accuracy')
ax.set_title('Feature Selection Methods — Comparison on Synthetic Data (5/30 informative)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0.6, 0.95)
ax.axhline(results_df.loc[0, 'Test Accuracy'], ls='--', color='gray', alpha=0.5,
           label=f'All features baseline: {results_df.loc[0, "Test Accuracy"]:.3f}')

for bar, val in zip(bars, results_df['Test Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.legend()
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 7. Stability of Feature Selection

A stable feature selector returns consistent feature sets across different data samples. We use cross-validation to measure how often each feature is selected.

In [ ]:
# Cross-validated stability analysis
cv = KFold(n_splits=10, shuffle=True, random_state=42)
selection_counts = np.zeros(X_cancer.shape[1])

for train_idx, _ in cv.split(X_cancer):
    X_fold = X_cancer.iloc[train_idx]
    y_fold = y_cancer[train_idx]

    selector = SelectFromModel(
        RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
        threshold='median'
    )
    selector.fit(X_fold, y_fold)
    selection_counts += selector.get_support().astype(int)

stability_df = pd.DataFrame({
    'feature': X_cancer.columns,
    'times_selected': selection_counts,
    'stability_pct': selection_counts / 10 * 100
}).sort_values('times_selected', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if v >= 8 else ('#f39c12' if v >= 5 else '#e74c3c')
          for v in stability_df['times_selected']]
ax.barh(range(len(stability_df)), stability_df['times_selected'],
        color=colors, edgecolor='white')
ax.set_yticks(range(len(stability_df)))
ax.set_yticklabels(stability_df['feature'])
ax.set_xlabel('Times Selected (out of 10 CV folds)')
ax.set_title('Feature Selection Stability Across 10 CV Folds', fontsize=13, fontweight='bold')
ax.axvline(5, ls='--', color='gray', alpha=0.5, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

stable_features = stability_df[stability_df['times_selected'] >= 7]['feature'].tolist()
print(f'Stable features (selected in ≥7/10 folds): {len(stable_features)}')
print(stable_features)

## 8. Real-World Example: Regression Feature Selection

Using the diabetes dataset to select features for a regression problem.

In [ ]:
# Load diabetes dataset
diabetes = load_diabetes()
X_diab = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diab = diabetes.target

X_diab_train, X_diab_test, y_diab_train, y_diab_test = train_test_split(
    X_diab, y_diab, test_size=0.25, random_state=42
)

print(f'Diabetes dataset: {X_diab.shape}')
print(f'Features: {list(X_diab.columns)}')
print(f'Target range: {y_diab.min():.1f} — {y_diab.max():.1f}')

In [ ]:
# Apply multiple methods to diabetes regression
methods_reg = {
    'All Features': None,
    'SelectKBest (f_regression)': SelectKBest(
        from sklearn.feature_selection import f_regression; f_regression(), k=4
    ),
    'RFE (LinearRegression)': RFE(LinearRegression(), n_features_to_select=4),
    'Lasso (alpha=1)': SelectFromModel(Lasso(alpha=1, random_state=42, max_iter=10000)),
    'SelectFromModel (RF)': SelectFromModel(
        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        threshold='median'
    )
}

results_reg = {}
for name, method in methods_reg.items():
    if name == 'All Features':
        lr = LinearRegression().fit(X_diab_train, y_diab_train)
        preds = lr.predict(X_diab_test)
    else:
        if isinstance(method, tuple):
            sel = SelectKBest(method[0], k=method[1])
        else:
            sel = method
        X_tr_sel = sel.fit_transform(X_diab_train, y_diab_train)
        X_te_sel = sel.transform(X_diab_test)
        lr = LinearRegression().fit(X_tr_sel, y_diab_train)
        preds = lr.predict(X_te_sel)

    results_reg[name] = {
        'R2': r2_score(y_diab_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_diab_test, preds))
    }

results_reg_df = pd.DataFrame(results_reg).T
print('Regression feature selection results:')
print(results_reg_df.round(4).to_string())

In [ ]:
# Model performance vs number of features (regression)
n_features_range = range(1, X_diab.shape[1] + 1)
r2_scores = []

for k in n_features_range:
    selector = SelectKBest(f_regression, k=k)
    X_sel = selector.fit_transform(X_diab_train, y_diab_train)
    lr = LinearRegression().fit(X_sel, y_diab_train)
    preds = lr.predict(selector.transform(X_diab_test))
    r2_scores.append(r2_score(y_diab_test, preds))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(n_features_range), r2_scores, marker='o', color='#3498db', lw=2, markersize=6)
ax.set_xlabel('Number of Features (by F-statistic rank)')
ax.set_ylabel('Test R² Score')
ax.set_title('Regression: Model Performance vs Number of Features', fontsize=13, fontweight='bold')
ax.axhline(0, ls='--', color='gray', alpha=0.5)
best_k = np.argmax(r2_scores) + 1
ax.axvline(best_k, ls='--', color='#e74c3c', alpha=0.7,
           label=f'Best k = {best_k} (R² = {r2_scores[best_k - 1]:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Best number of features: {best_k}')
print(f'Best features: {[diabetes.feature_names[i] for i in np.argsort(f_regression(X_diab_train, y_diab_train)[0])[::-1][:best_k]]}')

## 9. Summary

| Method | Type | Pros | Cons |
|--------|------|------|------|
| Variance Threshold | Filter | Ultra-fast, removes constants | No relationship with target |
| Correlation / Chi-Sq / ANOVA | Filter | Interpretable, fast | Only linear / specific relationships |
| Mutual Information | Filter | Captures non-linear | Requires more data, slower |
| Forward / Backward | Wrapper | Model-aware, good performance | Computationally expensive |
| RFE / RFECV | Wrapper | Recursive refinement, CV-stable | Can overfit to model choice |
| Lasso (L1) | Embedded | Built-in selection, no extra step | Struggles with correlated groups |
| Tree Importance | Embedded | Fast, captures interactions | Biased toward high-cardinality features |
| SelectFromModel | Embedded | Flexible, works with any estimator | Requires threshold tuning |

**General Recommendations:**
- Start with **filter methods** for a quick screen.
- Use **SelectFromModel** or **RFECV** for robust automated selection.
- Always embed selection inside a **Pipeline** to prevent data leakage.
- Check **stability** across CV folds before committing to a subset.
- Domain knowledge should ultimately guide feature removal.

## 10. Practice Exercises

1. **Variance Threshold Tuning:** On the breast cancer dataset, apply VarianceThreshold with thresholds [0.01, 0.05, 0.1, 0.2]. How many features remain? How does model accuracy change?

2. **Compare Chi-Square vs Mutual Information:** Create a synthetic dataset with 15 features (5 categorical, 5 numerical, 5 noisy binary). Use chi2 and mutual_info_classif. Which features does each method rank highest? Why the difference?

3. **RFE Step Size Experiment:** On the synthetic dataset from Section 6, run RFE with `step` values of 1, 3, and 5. Compare the selected features, runtime, and final accuracy. What trade-offs do you observe?

4. **Pipeline with Custom Threshold:** Build a Pipeline that scales data → selects features via SelectFromModel with threshold='0.5*mean' → trains an SVC. Evaluate with 5-fold CV. How does SVC performance compare to LogisticRegression?

5. **Stability Analysis for Lasso:** On the diabetes regression dataset, run Lasso with alpha=0.5 across 20 different random train/test splits. Record which features get non-zero coefficients. Which features are selected most consistently?